In [56]:
from Bio import SeqIO
import json

g1 = "../data/hg38/hg38.ml.fa"
g1_records = {}
for record in SeqIO.parse(g1, "fasta"):
    g1_records[record.id] = record.seq 

In [57]:
import json
g2_assembly = "../data/hg38/ncbi_dataset/data/GCF_000001405.26/sequence_report.jsonl"
data = []
with open(g2_assembly, "r") as f:
    for line in f:
        # Avoid empty lines
        if line.strip():  
            data.append(json.loads(line))

In [58]:
g2_id_to_g1_id = {}
for assembly in data:
    if assembly["ucscStyleName"] in g1_records:
        g2_id_to_g1_id[assembly["refseqAccession"]] = assembly["ucscStyleName"]

In [59]:
print(g2_id_to_g1_id)

{'NC_000001.11': 'chr1', 'NC_000002.12': 'chr2', 'NC_000003.12': 'chr3', 'NC_000004.12': 'chr4', 'NC_000005.10': 'chr5', 'NC_000006.12': 'chr6', 'NC_000007.14': 'chr7', 'NC_000008.11': 'chr8', 'NC_000009.12': 'chr9', 'NC_000010.11': 'chr10', 'NC_000011.10': 'chr11', 'NC_000012.12': 'chr12', 'NC_000013.11': 'chr13', 'NC_000014.9': 'chr14', 'NC_000015.10': 'chr15', 'NC_000016.10': 'chr16', 'NC_000017.11': 'chr17', 'NC_000018.10': 'chr18', 'NC_000019.10': 'chr19', 'NC_000020.11': 'chr20', 'NC_000021.9': 'chr21', 'NC_000022.11': 'chr22', 'NC_000023.11': 'chrX'}


In [60]:
from BCBio import GFF

gff_path = "../data/hg38/ncbi_dataset/data/GCF_000001405.26/genomic.gff"
limit_info = {
    "gff_id": list(g2_id_to_g1_id.keys())
}

reverse_orientation_features = {}

# Canonicalize so they're all in the + direction
with open(gff_path, "r") as f:
    for rec in GFF.parse(f, limit_info=limit_info):
        print(f"Parsing {rec.id}...")
        reverse_orientation_features[rec.id] = []
        for feature in rec.features:
            if feature.location.strand == -1:
               reverse_orientation_features[rec.id].append((feature.location.start, feature.location.end))

Parsing NC_000001.11...
Parsing NC_000002.12...
Parsing NC_000003.12...
Parsing NC_000004.12...
Parsing NC_000005.10...
Parsing NC_000006.12...
Parsing NC_000007.14...
Parsing NC_000008.11...
Parsing NC_000009.12...
Parsing NC_000010.11...
Parsing NC_000011.10...
Parsing NC_000012.12...
Parsing NC_000013.11...
Parsing NC_000014.9...
Parsing NC_000015.10...
Parsing NC_000016.10...
Parsing NC_000017.11...
Parsing NC_000018.10...
Parsing NC_000019.10...
Parsing NC_000020.11...
Parsing NC_000021.9...
Parsing NC_000022.11...
Parsing NC_000023.11...


In [61]:
# Take the complement, reverse and RC
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Seq import MutableSeq

def reverse(seq, start, end):
    seq[start:end] = seq[start:end][::-1]

def complement(seq, start, end):
    seq[start:end] = seq[start:end].complement()

def save_seqs_dict(seqs_dict, save_path):
    records = [
        SeqRecord(seq, id=name, description="")
        for name, seq in seqs_dict.items()
    ]
    SeqIO.write(records, save_path, "fasta")

test_seq = MutableSeq("AACCGGTT")
reverse(test_seq, 0, 4)
complement(test_seq, 0, 4)
print(test_seq)

GGTTGGTT


In [62]:
reverse_seqs = {}
complement_seqs = {}
rc_seqs = {}

for g2_seq_id, locations in reverse_orientation_features.items():
    g1_id = g2_id_to_g1_id[g2_seq_id]
    print(f"Parsing {g1_id}...")
    
    reverse_seq = MutableSeq(g1_records[g1_id])
    complement_seq = MutableSeq(g1_records[g1_id])
    rc_seq = MutableSeq(g1_records[g1_id])
    
    for (start, end) in locations:
        reverse(reverse_seq, start, end)
        complement(complement_seq, start, end)

        reverse(rc_seq, start, end)
        complement(rc_seq, start, end)

    reverse_seqs[g1_id] = reversed_seq
    complement_seqs[g1_id] = complement_seq
    rc_seqs[g1_id] = rc_seq

Parsing chr1...
Parsing chr2...
Parsing chr3...
Parsing chr4...
Parsing chr5...
Parsing chr6...
Parsing chr7...
Parsing chr8...
Parsing chr9...
Parsing chr10...
Parsing chr11...
Parsing chr12...
Parsing chr13...
Parsing chr14...
Parsing chr15...
Parsing chr16...
Parsing chr17...
Parsing chr18...
Parsing chr19...
Parsing chr20...
Parsing chr21...
Parsing chr22...
Parsing chrX...


In [66]:
print(reverse_orientation_features["NC_000001.11"][0])

(ExactPosition(14361), ExactPosition(29370))


In [73]:
print(g1_records["chr1"][14361])
print(complement_seqs["chr1"][14361])

T
A


In [75]:
out_path = "../data/parsed/"
save_seqs_dict(reverse_seqs, out_path + "reverse_seqs.fasta")
save_seqs_dict(complement_seqs, out_path + "complement_seqs.fasta")
save_seqs_dict(rc_seqs, out_path + "rc_seqs.fasta")